# Node Classification and Explainability on the PolBlogs Dataset

In this work, I use the `PolBlogs` dataset from PyTorch Geometric, which represents a network of political blogs.  
Each node corresponds to a political blog, edges represent hyperlinks between blogs, and the task is to classify each blog according to its political orientation.

I train and compare several Graph Neural Network (GNN) architectures for the node classification task, including GCN, GraphSAGE, and GAT.  
After training the models, I apply different explainability methods in order to better understand how the networks make their predictions and which graph structures are most influential for the classification of a selected node.

In [1]:
import torch
from torch_geometric.datasets import PolBlogs
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric as tg
from torch_geometric.utils import degree
import copy

C:\Users\maory\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading the PolBlogs Dataset

The following code loads the `PolBlogs` dataset from PyTorch Geometric.  
The dataset contains a graph of political blogs, where nodes represent blogs and edges represent hyperlinks between them.  
After loading the dataset, we extract the graph object and print basic information about the dataset and its graph structure.

The node classification task consists of two political classes: Liberal (Class 0) and Conservative (Class 1).  
The dataset is relatively balanced, containing 758 liberal blogs and 732 conservative blogs.

In [2]:
dataset = PolBlogs(root="data/PolBlogs")
data = dataset[0]

print(dataset)
print(data)

PolBlogs()
Data(edge_index=[2, 19025], y=[1490], num_nodes=1490)


In [3]:
print("Number of graphs:", len(dataset))
print("Number of nodes:", data.num_nodes)
print("Number of edges:", data.num_edges)
print("Has node features:", data.x is not None)
print("Node feature shape:", None if data.x is None else data.x.shape)
print("Labels shape:", data.y.shape)
print("Number of classes:", int(data.y.max().item() + 1))

unique_labels, counts = torch.unique(data.y, return_counts=True)

print("Label distribution:")
for label, count in zip(unique_labels.tolist(), counts.tolist()):
    name = "Liberal" if label == 0 else "Conservative"
    print(f"Class {label} ({name}): {count} nodes")

Number of graphs: 1
Number of nodes: 1490
Number of edges: 19025
Has node features: False
Node feature shape: None
Labels shape: torch.Size([1490])
Number of classes: 2
Label distribution:
Class 0 (Liberal): 758 nodes
Class 1 (Conservative): 732 nodes


In [4]:
# Create node features if missing
if data.x is None:
    deg = degree(data.edge_index[0], num_nodes=data.num_nodes)
    data.x = deg.view(-1, 1)

num_features = data.num_node_features
num_classes = int(data.y.max().item() + 1)

print("num_features:", num_features)
print("num_classes:", num_classes)

num_features: 1
num_classes: 2


# Examined GNN Architectures

In this section, several Graph Neural Network (GNN) architectures are examined for the node classification task on the PolBlogs dataset.  
The goal is to compare different message-passing mechanisms and later analyze how each architecture explains its predictions.

The examined architectures are:
- GCN (Graph Convolutional Network)
- GraphSAGE
- GAT (Graph Attention Network)

## 1. GCN (Graph Convolutional Network)

The first examined architecture is the Graph Convolutional Network (GCN).  
GCN performs message passing by aggregating information from neighboring nodes using a normalized graph convolution operation.  
This allows each node to gradually incorporate information from its local neighborhood.

The implemented model contains two graph convolution layers with a ReLU activation function and dropout regularization between them.  
The first layer learns hidden node representations, while the second layer produces the final classification scores for each node.

In [5]:
class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.3):
        super().__init__()
        self.conv1 = tg.nn.GCNConv(in_channels, hidden_channels)
        self.conv2 = tg.nn.GCNConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

## 2. GraphSAGE

The second examined architecture is GraphSAGE.  
Unlike GCN, GraphSAGE learns an aggregation function that combines information from neighboring nodes in a more flexible manner.  

The implemented model consists of two GraphSAGE convolution layers with ReLU activation and dropout regularization between them.  
During message passing, each node updates its representation by aggregating information from its neighbors and combining it with its own features.

In [6]:
class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.3):
        super().__init__()
        self.conv1 = tg.nn.SAGEConv(in_channels, hidden_channels)
        self.conv2 = tg.nn.SAGEConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

## 3. GAT (Graph Attention Network)

The third examined architecture is the Graph Attention Network (GAT).  
GAT introduces an attention mechanism into graph neural networks, allowing the model to assign different importance weights to different neighboring nodes during message passing.  
Instead of treating all neighbors equally, the model learns which neighbors are more influential for the target node representation.

The implemented model contains two GAT convolution layers with multi-head attention in the first layer, ELU activation, and dropout regularization.  
The use of attention coefficients also makes this architecture particularly interesting for explainability analysis, since the learned attention weights can provide insight into which neighboring nodes the model focuses on during prediction.

In [7]:
class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4, dropout=0.3):
        super().__init__()
        self.conv1 = tg.nn.GATConv(
            in_channels,
            hidden_channels,
            heads=heads,
            dropout=dropout
        )
        self.conv2 = tg.nn.GATConv(
            hidden_channels * heads,
            out_channels,
            heads=1,
            concat=False,
            dropout=dropout
        )
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

## Model Initialization

In the following step, the three examined GNN architectures are initialized with their corresponding hyperparameters.  
After initialization, the structure of each model is printed in order to inspect the layers and architecture configuration before training.

In [8]:
models = {
    "GCN": GCN(num_features, 32, num_classes),
    "GraphSAGE": GraphSAGE(num_features, 32, num_classes),
    "GAT": GAT(num_features, 8, num_classes, heads=4)
}

for name, model in models.items():
    print(name)
    print(model)
    print()

GCN
GCN(
  (conv1): GCNConv(1, 32)
  (conv2): GCNConv(32, 2)
)

GraphSAGE
GraphSAGE(
  (conv1): SAGEConv(1, 32, aggr=mean)
  (conv2): SAGEConv(32, 2, aggr=mean)
)

GAT
GAT(
  (conv1): GATConv(1, 8, heads=4)
  (conv2): GATConv(32, 2, heads=1)
)



## Train, Validation, and Test Split

Before training the models, the nodes in the graph are randomly divided into training, validation, and test sets.  
The function `create_masks` creates boolean masks that determine which nodes belong to each subset.

In this work:
- 60% of the nodes are used for training,
- 20% for validation,
- and 20% for testing.

The training set is used for optimizing the model parameters, the validation set is used for monitoring performance during training, and the test set is used for evaluating the final generalization performance of the trained models.

In [9]:
def create_masks(data, train_ratio=0.6, val_ratio=0.2):
    num_nodes = data.num_nodes
    perm = torch.randperm(num_nodes)

    train_end = int(train_ratio * num_nodes)
    val_end = int((train_ratio + val_ratio) * num_nodes)

    data.train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    data.val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    data.test_mask = torch.zeros(num_nodes, dtype=torch.bool)

    data.train_mask[perm[:train_end]] = True
    data.val_mask[perm[train_end:val_end]] = True
    data.test_mask[perm[val_end:]] = True

    return data

data = create_masks(data)

print(data.train_mask.sum(), data.val_mask.sum(), data.test_mask.sum())

tensor(894) tensor(298) tensor(298)


In [10]:
def train(model, data, optimizer, criterion):
    model.train()
    optimizer.zero_grad()

    out = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])

    loss.backward()
    optimizer.step()

    return loss.item()


@torch.no_grad()
def evaluate(model, data):
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)

    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        acc = (pred[mask] == data.y[mask]).float().mean().item()
        accs.append(acc)

    return accs

## Node Features and Model Initialization

The PolBlogs dataset does not contain intrinsic node features.  
Therefore, identity features were assigned to the nodes using an identity matrix representation (`torch.eye`).  
In this representation, each node receives a one-hot feature vector whose dimension equals the number of nodes in the graph.

After defining the node features, the number of input features and output classes is extracted from the dataset.  
The three examined GNN architectures (GCN, GraphSAGE, and GAT) are then initialized with their corresponding hidden dimensions and classification output size.

In [11]:
data.x = torch.eye(data.num_nodes)

num_features = data.num_node_features
num_classes = int(data.y.max().item() + 1)

models = {
    "GCN": GCN(num_features, 32, num_classes),
    "GraphSAGE": GraphSAGE(num_features, 32, num_classes),
    "GAT": GAT(num_features, 8, num_classes, heads=4)
}

## Model Training

Each examined GNN architecture is trained for 200 epochs using the Adam optimizer and cross-entropy loss for node classification.  
During training, the model performance is evaluated on the training, validation, and test subsets.

To avoid selecting a suboptimal model from the final epoch, the best model parameters are saved according to the highest validation accuracy achieved during training.  
After training is completed, the model is restored to its best-performing state and its final validation and test accuracies are stored for later comparison.

In [12]:
results = {}

for name, model in models.items():

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.01,
        weight_decay=5e-4
    )

    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0
    best_test_acc = 0

    # Save best model
    best_model_state = None

    print(f"Training {name}")

    for epoch in range(1, 201):

        loss = train(model, data, optimizer, criterion)

        train_acc, val_acc, test_acc = evaluate(model, data)

        # Save best model according to validation accuracy
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_test_acc = test_acc

            best_model_state = copy.deepcopy(model.state_dict())

        if epoch % 20 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Loss: {loss:.4f} | "
                f"Train: {train_acc:.3f} | "
                f"Val: {val_acc:.3f} | "
                f"Test: {test_acc:.3f}"
            )

    # Load best model back
    model.load_state_dict(best_model_state)

    results[name] = {
        "best_val_acc": best_val_acc,
        "best_test_acc": best_test_acc,
        "model": model
    }

    print()

Training GCN
Epoch 020 | Loss: 0.1630 | Train: 0.984 | Val: 0.906 | Test: 0.903
Epoch 040 | Loss: 0.0841 | Train: 0.992 | Val: 0.849 | Test: 0.856
Epoch 060 | Loss: 0.0719 | Train: 0.993 | Val: 0.889 | Test: 0.893
Epoch 080 | Loss: 0.0621 | Train: 0.993 | Val: 0.893 | Test: 0.896
Epoch 100 | Loss: 0.0546 | Train: 0.994 | Val: 0.879 | Test: 0.886
Epoch 120 | Loss: 0.0499 | Train: 0.996 | Val: 0.883 | Test: 0.883
Epoch 140 | Loss: 0.0465 | Train: 0.994 | Val: 0.876 | Test: 0.883
Epoch 160 | Loss: 0.0415 | Train: 0.994 | Val: 0.872 | Test: 0.886
Epoch 180 | Loss: 0.0429 | Train: 0.997 | Val: 0.872 | Test: 0.879
Epoch 200 | Loss: 0.0387 | Train: 0.997 | Val: 0.879 | Test: 0.879

Training GraphSAGE
Epoch 020 | Loss: 0.0765 | Train: 0.996 | Val: 0.869 | Test: 0.849
Epoch 040 | Loss: 0.0288 | Train: 1.000 | Val: 0.859 | Test: 0.839
Epoch 060 | Loss: 0.0304 | Train: 1.000 | Val: 0.856 | Test: 0.839
Epoch 080 | Loss: 0.0232 | Train: 1.000 | Val: 0.859 | Test: 0.839
Epoch 100 | Loss: 0.0210 | Tr

# Selecting a Node for Explainability Analysis

Before applying the explainability methods, a target node is selected for detailed analysis.  
The selected node is chosen from the test set and must satisfy two conditions:
- it is correctly classified by all examined models,
- and it has a relatively high graph degree (many neighboring nodes).

Choosing a high-degree node allows the explainability visualizations to contain a richer local neighborhood structure, making the comparison between different architectures and explanation methods more informative.

In [13]:
@torch.no_grad()
def find_good_node(models_dict, data):

    test_nodes = torch.where(data.test_mask)[0]

    for node_idx in test_nodes:

        all_correct = True

        for name, result in models_dict.items():

            model = result["model"]
            model.eval()

            out = model(data.x, data.edge_index)
            pred = out.argmax(dim=1)

            if pred[node_idx] != data.y[node_idx]:
                all_correct = False
                break

        if all_correct:
            return int(node_idx)

    return None


node_idx = find_good_node(results, data)

print("Chosen node:", node_idx)
print("True label:", data.y[node_idx].item())

Chosen node: 4
True label: 0


In [14]:
for name, result in results.items():
    model = result["model"]
    model.eval()

    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)

    print(name)
    print("Prediction:", pred[node_idx].item())
    print("True label:", data.y[node_idx].item())
    print()

GCN
Prediction: 0
True label: 0

GraphSAGE
Prediction: 0
True label: 0

GAT
Prediction: 0
True label: 0



In [15]:
@torch.no_grad()
def find_good_high_degree_node(results, data, min_degree=10):
    deg = torch.bincount(data.edge_index[0], minlength=data.num_nodes)
    test_nodes = torch.where(data.test_mask)[0]

    candidates = []

    for node_idx in test_nodes:
        all_correct = True

        for name, result in results.items():
            model = result["model"]
            model.eval()

            out = model(data.x, data.edge_index)
            pred = out.argmax(dim=1)

            if pred[node_idx] != data.y[node_idx]:
                all_correct = False
                break

        if all_correct and deg[node_idx] >= min_degree:
            candidates.append((int(node_idx), int(deg[node_idx]), int(data.y[node_idx])))

    candidates = sorted(candidates, key=lambda x: x[1], reverse=True)
    return candidates


candidates = find_good_high_degree_node(results, data, min_degree=10)

print("Top candidate nodes:")
for c in candidates[:10]:
    print(f"Node {c[0]} | Degree: {c[1]} | Label: {c[2]}")

Top candidate nodes:
Node 453 | Degree: 140 | Label: 0
Node 979 | Degree: 91 | Label: 1
Node 117 | Degree: 80 | Label: 0
Node 1184 | Degree: 79 | Label: 1
Node 764 | Degree: 74 | Label: 1
Node 416 | Degree: 73 | Label: 0
Node 611 | Degree: 71 | Label: 0
Node 190 | Degree: 67 | Label: 0
Node 1460 | Degree: 66 | Label: 1
Node 1107 | Degree: 60 | Label: 1


In [16]:
node_idx = candidates[0][0]

print("Chosen node:", node_idx)
print("Degree:", candidates[0][1])
print("True label:", data.y[node_idx].item())

Chosen node: 453
Degree: 140
True label: 0


# Explainability Analysis

After training the GNN models, explainability methods are applied in order to better understand how the networks make their predictions.  
The goal is to identify which graph structures and neighboring nodes are most influential for the classification of the selected target node.

In this work, the explanations focus mainly on graph topology and important connections between nodes, since the dataset does not contain intrinsic semantic node features and identity features were used instead.

The first explainability method examined is `GNNExplainer`, which attempts to learn a compact subgraph that best explains the prediction made by the model for the selected node.

In [17]:
from torch_geometric.explain import Explainer
from torch_geometric.explain.algorithm import GNNExplainer
import torch_geometric as tg
from IPython.display import HTML
from IPython.display import IFrame, display
import time

In [18]:
gcn_model = results["GCN"]["model"]

explainer = Explainer(
    model=gcn_model,
    algorithm=GNNExplainer(epochs=200),

    explanation_type='model',

    node_mask_type='attributes',
    edge_mask_type='object',

    model_config=dict(
        mode='multiclass_classification',
        task_level='node',
        return_type='raw',
    ),
)

explanation = explainer(
    data.x,
    data.edge_index,
    index=node_idx,
)

In [19]:
model = results["GCN"]["model"]
model.eval()

gnn_explainer = tg.explain.Explainer(
    model=model,
    algorithm=tg.explain.GNNExplainer(epochs=50),
    explanation_type="model",
    node_mask_type="attributes",
    edge_mask_type="object",
    model_config=dict(
        mode="multiclass_classification",
        task_level="node",
        return_type="raw",
    ),
)

gnn_explanation = gnn_explainer(
    data.x,
    data.edge_index,
    index=node_idx
)

gnn_node_scores = gnn_explanation.node_mask.max(dim=1).values

print("GNNExplainer node mask shape:", tuple(gnn_explanation.node_mask.shape))
print("GNNExplainer edge mask shape:", tuple(gnn_explanation.edge_mask.shape))

GNNExplainer node mask shape: (1490, 1490)
GNNExplainer edge mask shape: (19025,)


In [20]:
from collections import defaultdict
import torch
import torch as tch
import torch_geometric as tg
import pyvis.network as pyv_n
from IPython.display import HTML


def collapse_edge_scores(edge_index, edge_scores):
    grouped = defaultdict(list)
    for (src, dst), score in zip(edge_index.t().tolist(), edge_scores.tolist()):
        key = tuple(sorted((src, dst)))
        grouped[key].append(float(score))
    return {key: sum(values) / len(values) for key, values in grouped.items()}


def normalize_dict(score_dict):
    if not score_dict:
        return {}
    values = list(score_dict.values())
    vmin, vmax = min(values), max(values)
    if abs(vmax - vmin) < 1e-12:
        return {key: 1.0 for key in score_dict}
    return {key: (value - vmin) / (vmax - vmin) for key, value in score_dict.items()}


def normalize_tensor(values):
    values = values.detach().float()
    vmin, vmax = float(values.min()), float(values.max())
    if abs(vmax - vmin) < 1e-12:
        return tch.ones_like(values)
    return (values - vmin) / (vmax - vmin)


def pyvis_explanation(graph: tg.data.Data, edge_scores, node_score, focus_node, top_k=30):
    pyv_graph = pyv_n.Network(
        directed=False,
        notebook=True,
        cdn_resources="remote",
        height="700px",
        width="100%"
    )

    collapsed = normalize_dict(collapse_edge_scores(graph.edge_index, edge_scores))
    top_edges = sorted(collapsed.items(), key=lambda x: x[1], reverse=True)[:top_k]

    node_score = normalize_tensor(node_score)

    nodes_exists = set()
    for (u, v), score in top_edges:
        nodes_exists.add(int(u))
        nodes_exists.add(int(v))

    nodes_exists.add(int(focus_node))

    for n_id in nodes_exists:
        n_id = int(n_id)

        if n_id == focus_node:
            color = "#FFAA55"
            shape = "star"
            size = 35
        else:
            color = "#AAAAAA"
            shape = "circle"
            size = float(node_score[n_id]) * 25 + 8

        title = f"Node: {n_id}<br>Strength: {float(node_score[n_id]):.3f}<br>Label: {int(graph.y[n_id])}"

        pyv_graph.add_node(
            n_id,
            label=str(n_id),
            title=title,
            color=color,
            shape=shape,
            size=size
        )

    for (u, v), score in top_edges:
        pyv_graph.add_edge(
            int(u),
            int(v),
            width=float(score) * 10 + 1,
            title=f"Edge score: {float(score):.3f}"
        )

    pyv_graph.force_atlas_2based()
    return pyv_graph


def run_gnn_explainer_and_visualize(model, model_name, data, node_idx, top_k=200):
    model.eval()

    gnn_explainer = tg.explain.Explainer(
        model=model,
        algorithm=tg.explain.GNNExplainer(epochs=50),
        explanation_type="model",
        node_mask_type="attributes",
        edge_mask_type="object",
        model_config=dict(
            mode="multiclass_classification",
            task_level="node",
            return_type="raw",
        ),
    )

    gnn_explanation = gnn_explainer(
        data.x,
        data.edge_index,
        index=node_idx
    )

    gnn_node_scores = gnn_explanation.node_mask.max(dim=1).values

    print(f"{model_name} explanation")
    print("GNNExplainer node mask shape:", tuple(gnn_explanation.node_mask.shape))
    print("GNNExplainer edge mask shape:", tuple(gnn_explanation.edge_mask.shape))

    top_feature_ids = torch.topk(
        gnn_explanation.node_mask[node_idx],
        k=min(5, gnn_explanation.node_mask.shape[1])
    ).indices.tolist()

    print(f"Top feature indices for node {node_idx}: {top_feature_ids}")

    file_name = f"{model_name.lower()}_gnn_explainer_node_{node_idx}_{int(time.time()*1000)}.html"
    
    expl_graph = pyvis_explanation(
        graph=data,
        edge_scores=gnn_explanation.edge_mask,
        node_score=gnn_node_scores,
        focus_node=node_idx,
        top_k=top_k
    )

    expl_graph.save_graph(file_name)
    
    display(IFrame(src=file_name, width="100%", height="750px"))
    
    return gnn_explanation

In [21]:
explanation_gcn = run_gnn_explainer_and_visualize(
    results["GCN"]["model"],
    "GCN",
    data,
    node_idx,
    top_k=50
)

GCN explanation
GNNExplainer node mask shape: (1490, 1490)
GNNExplainer edge mask shape: (19025,)
Top feature indices for node 453: [453, 1, 3, 4, 0]


In [22]:
explanation_sage = run_gnn_explainer_and_visualize(
    results["GraphSAGE"]["model"],
    "GraphSAGE",
    data,
    node_idx,
    top_k=50
)

GraphSAGE explanation
GNNExplainer node mask shape: (1490, 1490)
GNNExplainer edge mask shape: (19025,)
Top feature indices for node 453: [453, 1, 3, 4, 0]


In [23]:
explanation_gat = run_gnn_explainer_and_visualize(
    results["GAT"]["model"],
    "GAT",
    data,
    node_idx,
    top_k=50
)

GAT explanation
GNNExplainer node mask shape: (1490, 1490)
GNNExplainer edge mask shape: (19025,)
Top feature indices for node 453: [453, 1, 3, 4, 0]


## Attention-Based Explainability for GAT

In addition to GNNExplainer, an additional explainability method called `AttentionExplainer` is applied to the GAT architecture.  
Unlike GNNExplainer, which learns an explanatory subgraph through optimization, AttentionExplainer directly uses the attention coefficients learned internally by the GAT model during message passing.

This allows the analysis of which neighboring nodes received the highest attention weights during the prediction process, providing insight into how the GAT architecture focuses on different parts of the graph.

In [24]:
def run_attention_explainer(model, model_name, data, node_idx, top_k=30):
    model.eval()

    explainer = tg.explain.Explainer(
        model=model,
        algorithm=tg.explain.AttentionExplainer(),
        explanation_type="model",
        edge_mask_type="object",
        model_config=dict(
            mode="multiclass_classification",
            task_level="node",
            return_type="raw",
        ),
    )

    explanation = explainer(
        data.x,
        data.edge_index,
        index=node_idx
    )

    node_scores = torch.zeros(data.num_nodes)

    for (u, v), score in zip(data.edge_index.t(), explanation.edge_mask):
        node_scores[int(u)] += score.detach().cpu()
        node_scores[int(v)] += score.detach().cpu()

    print(f"{model_name} | AttentionExplainer")
    print("Edge mask shape:", tuple(explanation.edge_mask.shape))

    expl_graph = pyvis_explanation(
        graph=data,
        edge_scores=explanation.edge_mask,
        node_score=node_scores,
        focus_node=node_idx,
        top_k=top_k
    )

    file_name = f"{model_name.lower()}_attentionexplainer_node_{node_idx}_{int(time.time()*1000)}.html"

    expl_graph.save_graph(file_name)

    display(IFrame(src=file_name, width="100%", height="750px"))

    return explanation

In [27]:
attention_explanation = run_attention_explainer(
    model=results["GAT"]["model"],
    model_name="GAT",
    data=data,
    node_idx=node_idx,
    top_k=30
)

GAT | AttentionExplainer
Edge mask shape: (19025,)


# Results and Discussion

## General Explanation of the Results

Since the selected target node may change between different executions of the notebook, the following discussion focuses on one representative example obtained during the experiments.

In this example, node 453 was selected for the explainability analysis.  
This node was correctly classified as Liberal by all examined models, and its true label is also Liberal.  
In addition, the node has a relatively high degree of 140, making it suitable for analyzing and comparing the explanatory subgraphs produced by the different architectures and explainability methods.

The visualization figures generated during the explainability analysis are attached in the repository (4 graph images in total) in the file `HW4 - explainer visualisations.pdf`.

## Comparison Between the Different Models Using GNNExplainer

All three examined architectures were analyzed using GNNExplainer.  
The main observed difference between the models is the number of edges incident to node 453 that were identified as important among the top-50 explanatory edges.

For node 453:
- GCN highlighted 22 incident edges,
- GraphSAGE highlighted 7 incident edges,
- and GAT highlighted 4 incident edges.

These differences may be related to the internal message-passing mechanisms of the models themselves.  
GCN aggregates information from neighboring nodes using a normalized graph convolution operation, which often leads to explanations that rely on a broader local neighborhood around the target node.

GraphSAGE uses a learned aggregation function, which may produce more selective neighborhood representations and therefore fewer directly important edges in the explanation.

GAT introduces an attention mechanism that assigns different importance weights to neighboring nodes.  
As a result, the model may focus only on a small subset of highly influential neighbors, leading to a smaller number of important incident edges identified by GNNExplainer.

In [29]:
import networkx as nx

# Convert PyG graph to NetworkX graph
G = nx.Graph()

edge_list = data.edge_index.t().tolist()
G.add_edges_from(edge_list)

source_node = 453
target_node = 404

# Check shortest path distance
distance = nx.shortest_path_length(G, source=source_node, target=target_node)

print(f"Shortest path distance between {source_node} and {target_node}: {distance}")

Shortest path distance between 453 and 404: 1


Another interesting observation is that several nodes appear in the explanatory subgraphs of multiple models.  
One particularly notable example is node 404.

In the original graph, node 404 is directly connected to node 453 (shortest-path distance of 1).  
However, its role in the explanatory subgraphs differs significantly between the examined architectures:
- In GCN, node 404 does not appear in the explanatory neighborhood of node 453,
- In GraphSAGE, it appears at distance 2,
- and in GAT, it appears at distance 1.

This suggests that the different architectures rely on different subsets of the local neighborhood when generating predictions.  
In particular, the attention mechanism of GAT may emphasize certain highly relevant neighboring nodes that are ignored or assigned lower importance by other architectures.

## Comparison Between Different Explainability Methods for GAT

The GAT architecture was analyzed using two different explainability approaches: `GNNExplainer` and `AttentionExplainer`.

`GNNExplainer` attempts to learn an explanatory subgraph through an optimization process, searching for a compact set of nodes and edges that best explains the model prediction for the selected node.

In contrast, `AttentionExplainer` directly uses the attention coefficients learned internally by the GAT model during message passing.  
Instead of learning a new explanatory structure, it extracts the neighbor importance information already encoded in the model itself.

A notable structural difference can be observed between the resulting explanatory graphs.  
The explanatory graph produced by GNNExplainer consists of a relatively small number of connected components, where each component is fairly large and structurally rich.  
On the other hand, the AttentionExplainer graph contains many small connected components, most of which consist of only a single edge.

One possible explanation is that GNNExplainer explicitly searches for compact subgraphs that collectively preserve the model prediction, naturally encouraging more connected explanatory structures.  
In contrast, AttentionExplainer reflects local attention scores independently for many edges, which may lead to sparse and fragmented explanatory patterns focused on highly weighted individual neighbor relations.